In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chat_models.base import init_chat_model
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import pandas as pd

load_dotenv()

True

In [41]:
# OpenAI:
# llm = init_chat_model("openai:gpt-3.5-turbo")

# embedding_model = "text-embedding-3-small" # outputs 1536 dimensions by default
# embedding = OpenAIEmbeddings(model=embedding_model)

# Ollama:
llm = ChatOllama(model="gemma3:12b")
embedding = OllamaEmbeddings(model="nomic-embed-text")

In [42]:

fnx_doc = []

try:
    df = pd.read_excel(
    "../DataIngestParsing/data/csv_excel/users_table.xlsx"
    )

    for _, row in df.iterrows():

        text = f"""
        פוליסה מספר {row.get('מספר הפוליסה', '')},
        הפוליסה מנוהלת על ידי {row.get('שם סוכן', '')},
        נוצרה על ידי {row.get('נציג', '')},
        עודכנה על ידי {row.get('עודכן ע"י', '')},
        זוהי פוליסה {"לא פעילה" if row.get("IsActive") == 0 else "פעילה"}.
        """

        # Clean metadata
        metadata = {}

        for key, value in row.items():

            # Skip NaN
            if pd.isna(value):
                continue

            # Convert timestamps to string
            if isinstance(value, pd.Timestamp):
                value = value.isoformat()

            # Convert numpy types to native Python
            if hasattr(value, "item"):
                value = value.item()

            metadata[str(key)] = value

        fnx_doc.append(
            Document(
                page_content=text,
                metadata=metadata
            )
        )

    print(f"The embedding data will look like that: {fnx_doc[0]}")
except Exception as e:
    print(f"Error loading PyPDF: {e}")


The embedding data will look like that: page_content='
        פוליסה מספר 1136442,
        הפוליסה מנוהלת על ידי SMART הפניקס,
        נוצרה על ידי ישי אמסלם,
        עודכנה על ידי שלו קריו,
        זוהי פוליסה פעילה.
        ' metadata={'Id': 29130, 'PolicyId': 250961121136442, 'שנת חיתום': 25, 'BranchNumber': 96, 'Anaf': 112, 'מספר הפוליסה': 1136442, 'מזהה לקוח': 28871, 'תאריך תחילת הפוליסה': '2025-05-17T00:00:00', 'תאריך סיום הפוליסה': '2026-05-11T14:42:00.787000', 'סטטוס הפוליסה': 2, 'StatusDate': '2026-05-11T14:42:00.787000', 'מספר סוכן': 64096, 'שם סוכן': 'SMART הפניקס', 'InstanceId': 7682268845, 'ExternalStatus': 2.0, 'תאריך יצירת הרשומה': '2025-05-17T22:12:56.713000', 'נציג': 'ישי אמסלם', 'CreatorType': 1, 'עודכן ע"י': 'שלו קריו', 'UpdatorType': 2.0, 'UpdateDate': '2026-05-11T14:42:00.787000', 'IsActive': 1, 'IsCancelled': 1.0, 'OriginalPolicyEndDate': '2026-05-31T23:59:59', 'PolicyCancellationReasonCode': 1.0, 'IsAgent': 0, 'UpdatedByUserName': 'usf01266', 'NumberOfDenials': 

In [7]:
 # Vector store: ChromaDB (Chroma.from_documents) automatically infers the dimension from the first embedding it receives — you don't need to configure it manually. It adapts to whatever model you pass.

persist_directory = "./chroma_db_store"
vector_store = Chroma.from_documents(
    documents=fnx_doc,
    embedding=embedding,
    persist_directory=persist_directory, # This will create chroma_db_store folder for the vector store with all the data
    collection_name="rag_collection"
)

print(f"Created vector store with {len(vector_store)} vectors")

Created vector store with 2346 vectors


In [43]:
# Similarity TEST:

query = "תביא לי בבקשה פרטים על פוליסה מספר 1138938"

# similar_docs = vector_store.similarity_search(query, k=2)
similar_docs_with_scores = vector_store.similarity_search_with_score(query, k=2)

# scores with ChromaDB:
# Closer to 0 = Similar result
# Zero means identical vectors

# print(similar_docs[0])
# print("---------")
print(similar_docs_with_scores[0])

(Document(metadata={'מזהה לקוח': 29455, 'שם סוכן': 'SMART הפניקס', 'InstanceId': 1711462356, 'NumberOfClaims': 0.0, 'NumberOfDenials': 0.0, 'StatusDate': '2026-05-11T12:51:06.100000', 'IsActive': 1, 'PolicyId': 250961121138938, 'FnxInsurancePolicyStatus': 1.0, 'מספר סוכן': 64096, 'נציג': 'נתנאל ויצמן', 'UpdateDate': '2026-05-11T12:51:06.100000', 'UpdatorType': 2.0, 'OriginalPolicyEndDate': '2026-05-31T23:59:59', 'עודכן ע"י': 'הודיה מכובד', 'UpdatedByUserName': 'usf00271', 'BranchNumber': 96, 'CreatedByUserName': 'usf89483', 'תאריך יצירת הרשומה': '2025-05-25T09:05:37.693000', 'Id': 29724, 'תאריך תחילת הפוליסה': '2025-05-25T00:00:00', 'ExternalStatus': 2.0, 'תאריך סיום הפוליסה': '2026-05-11T12:51:06.100000', 'מספר הפוליסה': 1138938, 'PolicyCancellationReasonCode': 1.0, 'שנת חיתום': 25, 'IsCancelled': 1.0, 'Anaf': 112, 'IsAgent': 0, 'סטטוס הפוליסה': 2, 'CreatorType': 7}, page_content='\n        פוליסה מספר 1138938,\n        הפוליסה מנוהלת על ידי SMART הפניקס,\n        נוצרה על ידי נתנאל ו

In [44]:
custom_prompt = """
        אתה נציג שירות מקצועי של חברת הפניקס.

        המטרה שלך היא לספק מידע מדויק, ברור ואדיב ללקוחות החברה בנושאי ביטוח, פוליסות, סטטוס פוליסה, תביעות, חידושים, פרטי סוכן, ומידע כללי הקשור לשירותי החברה.

        הנחיות עבודה:

        * תמיד השב בעברית ברורה ומקצועית.
        * השתמש בטון שירותי, סבלני ומכבד.
        * אם מידע חסר או לא נמצא במערכת — ציין זאת בצורה ברורה.
        * אל תמציא מידע שאינו קיים בנתונים שסופקו לך.
        * כאשר לקוח שואל על פוליסה, נסה להחזיר:

          * מספר פוליסה
          * סטטוס פוליסה
          * תאריכי התחלה וסיום
          * שם הסוכן המטפל
          * האם הפוליסה פעילה
        * אם נמצאו מספר תוצאות דומות — בקש פרטים נוספים לזיהוי.
        * במקרה של שאלה שאינה קשורה להפניקס או למידע הקיים במערכת, השב בנימוס שאינך יכול לסייע בנושא זה.
        * אל תחשוף מידע רגיש שאינו נדרש לצורך המענה.
        * נסח תשובות בצורה טבעית ואנושית, ולא כרשימת נתונים טכנית בלבד.

        דוגמאות לסגנון תשובה:
        * "הפוליסה שלך פעילה עד לתאריך 11/05/2026."
        * "הסוכן המטפל בפוליסה הוא SMART הפניקס."
        * "לא נמצאה פוליסה תואמת לפי הפרטים שסופקו."


        Context: {context}
    """

In [45]:
import re
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from pydantic import Field
from typing import Any

class HybridRetriever(BaseRetriever):
    vector_store: Any = Field(...)
    k: int = Field(default=3)

    def _get_relevant_documents(self, query: str) -> list[Document]:
        policy_match = re.search(r'\b(\d{6,10})\b', query)

        # 1. The regex + metadata filter is a structured exact lookup
        if policy_match:
            policy_number = int(policy_match.group(1))
            results = self.vector_store.get(
                where={"מספר הפוליסה": policy_number} #  where filter doesn't touch the text at all — it queries the *metadata* dictionary, not page_content.
            )
            docs = [
                Document(page_content=pc, metadata=meta)
                for pc, meta in zip(results["documents"], results["metadatas"])
            ]
            if docs:
                return docs

        return self.vector_store.similarity_search(query, k=self.k)  # 2. Dense similarity search (semantic search)

retriever = HybridRetriever(vector_store=vector_store)


In [46]:
prompt = ChatPromptTemplate.from_messages([
    ("system", custom_prompt),
    ("human", "{input}")
])

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n        אתה נציג שירות מקצועי של חברת הפניקס.\n\n        המטרה שלך היא לספק מידע מדויק, ברור ואדיב ללקוחות החברה בנושאי ביטוח, פוליסות, סטטוס פוליסה, תביעות, חידושים, פרטי סוכן, ומידע כללי הקשור לשירותי החברה.\n\n        הנחיות עבודה:\n\n        * תמיד השב בעברית ברורה ומקצועית.\n        * השתמש בטון שירותי, סבלני ומכבד.\n        * אם מידע חסר או לא נמצא במערכת — ציין זאת בצורה ברורה.\n        * אל תמציא מידע שאינו קיים בנתונים שסופקו לך.\n        * כאשר לקוח שואל על פוליסה, נסה להחזיר:\n\n          * מספר פוליסה\n          * סטטוס פוליסה\n          * תאריכי התחלה וסיום\n          * שם הסוכן המטפל\n          * האם הפוליסה פעילה\n        * אם נמצאו מספר תוצאות דומות — בקש פרטים נוספים לזיהוי.\n        * במקרה של שאלה שאינה קשורה להפניקס או למידע הקיים במערכ

In [47]:
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)


In [51]:
response = rag_chain.invoke({
    "input": "תביא לי את כל הפרטים שיש לך על פוליסה מספר 1014320",
})

print(response.get("answer"))

שלום רב,

אשמח לספק לך את הפרטים שיש לי על פוליסה מספר 1014320.

*   **מספר פוליסה:** 1014320
*   **סטטוס פוליסה:** פעילה
*   **סוכן מנהל:** נח סוכנות ביטוח בע"מ
*   **אנשים מעורבים:** הפוליסה נוצרה על ידי אילון אוחיון ועדכנה אושר כהן.

אלו הפרטים שנמצאו במערכת. אם תרצה לדעת פרטים נוספים, כגון תאריכי התחלה וסיום הפוליסה או פרטים נוספים לגבי הכיסוי, אנא ציין זאת.
